## Example 1: Basic imread and imwrite with Format Support

Demonstrates core imageio.v3 functions: imread and imwrite with various formats.

In [ ]:
# Example 1: Basic imread and imwrite with format support
try:
    import imageio.v3 as iio
    import imageio
    import numpy as np
    import tempfile
    import os
    
    print("=== ImageIO Version and Capabilities ===")
    print(f"ImageIO version: {imageio.__version__}")
    
    # List supported formats (from main imageio module)
    formats = imageio.formats
    print(f"Total supported formats: {len(formats)}")
    print(f"Sample formats: {[f.name for f in list(formats)[:8]]}")
    
    print("\n=== Create Synthetic Image Data ===")
    # Create synthetic RGB image
    height, width = 100, 150
    image_rgb = np.zeros((height, width, 3), dtype=np.uint8)
    
    # Add colored patterns
    image_rgb[:, :50, 0] = 255  # Red section
    image_rgb[:, 50:100, 1] = 255  # Green section  
    image_rgb[:, 100:, 2] = 255  # Blue section
    
    # Add gradient pattern
    for i in range(height):
        image_rgb[i, :, :] = (image_rgb[i, :, :].astype(float) * (i / height)).astype(np.uint8)
    
    print(f"Created synthetic image: {image_rgb.shape}, dtype: {image_rgb.dtype}")
    print(f"Pixel value range: [{image_rgb.min()}, {image_rgb.max()}]")
    
    print("\n=== Using imread and imwrite (unique to imageio) ===")
    # Write image in different formats using imwrite
    with tempfile.TemporaryDirectory() as temp_dir:
        # Write PNG
        png_path = os.path.join(temp_dir, 'test.png')
        iio.imwrite(png_path, image_rgb)
        
        # Write JPEG with quality parameter
        jpg_path = os.path.join(temp_dir, 'test.jpg')
        iio.imwrite(jpg_path, image_rgb, quality=90)
        
        # Read images back using imread
        png_read = iio.imread(png_path)
        jpg_read = iio.imread(jpg_path)
        
        print(f"PNG: Original shape {image_rgb.shape} -> Read shape {png_read.shape}")
        print(f"JPEG: Original shape {image_rgb.shape} -> Read shape {jpg_read.shape}")
        print(f"PNG lossless: {np.array_equal(image_rgb, png_read)}")
        print(f"JPEG lossy difference: {np.mean(np.abs(image_rgb.astype(float) - jpg_read.astype(float))):.2f}")
        
        # Demonstrate imread with index for multi-page
        print(f"\nOriginal image channel 0 mean: {image_rgb[:,:,0].mean():.1f}")
        print(f"PNG read channel 0 mean: {png_read[:,:,0].mean():.1f}")
    
    print("Example 1: imread and imwrite test passed")
    
except Exception as e:
    print(f"Example 1: Error in basic image I/O test: {e}")
    import traceback
    traceback.print_exc()
    raise

## Example 2: Animated GIF Creation and imiter for Frame Iteration

Demonstrates animated image creation and the imiter function for memory-efficient frame iteration.

In [ ]:
# Example 2: Animated GIF creation and imiter for frame iteration
try:
    import imageio.v3 as iio
    import numpy as np
    import tempfile
    import os
    
    print("=== Create Synthetic Video Sequence ===")
    # Create animated sequence
    frames = []
    n_frames = 10
    height, width = 64, 64
    
    for frame_idx in range(n_frames):
        # Create frame with moving pattern
        frame = np.zeros((height, width, 3), dtype=np.uint8)
        
        # Moving circle
        center_x = int((frame_idx / n_frames) * width)
        center_y = height // 2
        
        # Create circular pattern
        y, x = np.ogrid[:height, :width]
        mask = (x - center_x)**2 + (y - center_y)**2 <= 100
        
        frame[mask] = [255, 128, 64]  # Orange circle
        
        # Add frame number as intensity variation
        frame[:, :, 1] = np.clip(frame[:, :, 1].astype(int) + (frame_idx * 20), 0, 255).astype(np.uint8)
        
        frames.append(frame)
    
    print(f"Created {len(frames)} frames of shape {frames[0].shape}")
    
    print("\n=== Animated GIF Creation with imwrite ===")
    with tempfile.TemporaryDirectory() as temp_dir:
        gif_path = os.path.join(temp_dir, 'animation.gif')
        
        # Write animated GIF (imwrite handles sequences)
        iio.imwrite(gif_path, frames, duration=200, loop=0)
        print(f"Created animated GIF: {gif_path}")
        
        # Read animated GIF back
        gif_frames = iio.imread(gif_path)
        print(f"Read GIF shape: {gif_frames.shape}")
        
        # Verify frame content
        print(f"Original frames: {len(frames)}")
        print(f"GIF frames: {gif_frames.shape[0]}")
        
        print("\n=== Frame-by-Frame Iteration with imiter ===")
        # Use imiter for memory-efficient frame iteration (unique to imageio v3)
        processed_frames = []
        
        for i, frame in enumerate(iio.imiter(gif_path)):
            # Apply simple processing (brightness adjustment)
            processed = np.clip(frame.astype(float) * 1.2, 0, 255).astype(np.uint8)
            processed_frames.append(processed)
            
            if i < 3:  # Show first few frames
                print(f"Frame {i}: shape={frame.shape}, range=[{frame.min()}, {frame.max()}] -> processed=[{processed.min()}, {processed.max()}]")
        
        print(f"Processed {len(processed_frames)} frames using imiter")
        
        # Save processed animation
        processed_gif_path = os.path.join(temp_dir, 'processed_animation.gif')
        iio.imwrite(processed_gif_path, processed_frames, duration=150)
        print(f"Saved processed animation with {len(processed_frames)} frames")
        
        # Verify file sizes
        orig_size = os.path.getsize(gif_path)
        proc_size = os.path.getsize(processed_gif_path)
        print(f"Original: {orig_size} bytes, Processed: {proc_size} bytes")
    
    print("Example 2: Animated GIF and imiter test passed")
    
except Exception as e:
    print(f"Example 2: Error in video processing test: {e}")
    import traceback
    traceback.print_exc()
    raise

## Example 3: Advanced Image Processing and Metadata

Demonstrate advanced image processing, metadata handling, and multi-channel images.

In [ ]:
# Example 3: Advanced image processing and metadata
try:
    import imageio.v3 as iio
    import numpy as np
    import tempfile
    import os
    
    print("=== Multi-Channel and Scientific Images ===")
    # Create multi-channel scientific image (e.g., microscopy data)
    height, width, channels = 128, 128, 4
    
    # Simulate different imaging channels
    multi_channel = np.zeros((height, width, channels), dtype=np.uint16)  # 16-bit data
    
    # Channel 0: DAPI (nuclei) - circular patterns
    y, x = np.ogrid[:height, :width]
    for i in range(3):
        center_x = 30 + i * 40
        center_y = 30 + i * 30
        mask = (x - center_x)**2 + (y - center_y)**2 <= 200
        multi_channel[mask, 0] = 40000 + i * 5000
    
    # Channel 1: GFP - elongated structures
    multi_channel[40:90, :, 1] = 35000
    multi_channel[:, 40:90, 1] += 20000
    
    # Channel 2: RFP - scattered points
    np.random.seed(42)
    for _ in range(20):
        px, py = np.random.randint(0, height), np.random.randint(0, width)
        multi_channel[max(0, px-5):min(height, px+5), 
                     max(0, py-5):min(width, py+5), 2] = 30000
    
    # Channel 3: Brightfield
    multi_channel[:, :, 3] = 25000 + np.random.normal(0, 2000, (height, width))
    
    print(f"Created multi-channel image: {multi_channel.shape}, dtype: {multi_channel.dtype}")
    print(f"Value ranges per channel: {[f'Ch{i}: [{multi_channel[:,:,i].min()}, {multi_channel[:,:,i].max()}]' for i in range(channels)]}")
    
    print("\n=== Metadata and Properties ===")
    with tempfile.TemporaryDirectory() as temp_dir:
        # Save with metadata
        tiff_path = os.path.join(temp_dir, 'multichannel.tiff')
        
        # TIFF supports metadata
        metadata = {
            'description': 'Multi-channel microscopy image',
            'channels': ['DAPI', 'GFP', 'RFP', 'Brightfield'],
            'pixel_size': 0.065,  # microns per pixel
            'objective': '40x',
            'exposure_times': [100, 200, 150, 50]  # milliseconds
        }
        
        iio.imwrite(tiff_path, multi_channel, metadata=metadata)
        print(f"Saved TIFF with metadata")
        
        # Read back and check metadata
        read_image = iio.imread(tiff_path)
        read_metadata = iio.immeta(tiff_path)
        
        print(f"Read image shape: {read_image.shape}")
        print(f"Metadata keys: {list(read_metadata.keys())[:5]}")
        
        # Check if our metadata is preserved
        if 'description' in read_metadata:
            print(f"Description preserved: {read_metadata['description']}")
        
    print("\n=== Image Statistics and Analysis ===")
    # Compute channel statistics
    for ch in range(channels):
        channel_data = multi_channel[:, :, ch]
        stats = {
            'mean': np.mean(channel_data),
            'std': np.std(channel_data),
            'min': np.min(channel_data),
            'max': np.max(channel_data),
            'non_zero_pixels': np.count_nonzero(channel_data)
        }
        print(f"Channel {ch}: mean={stats['mean']:.0f}, std={stats['std']:.0f}, non-zero={stats['non_zero_pixels']}")
    
    # Cross-correlation between channels
    ch0_flat = multi_channel[:, :, 0].flatten()
    ch1_flat = multi_channel[:, :, 1].flatten()
    correlation = np.corrcoef(ch0_flat, ch1_flat)[0, 1]
    print(f"Correlation between channels 0-1: {correlation:.3f}")
    
    print("Example 3: Advanced processing and metadata test passed")
    
except Exception as e:
    print(f"Example 3: Error in advanced processing test: {e}")
    import traceback
    traceback.print_exc()
    raise

## Example 4: Memory-Efficient Streaming with imiter

Demonstrates imiter for memory-efficient frame iteration and selective frame access with index parameter.

In [ ]:
# Example 4: Memory-efficient streaming with imiter
try:
    import imageio.v3 as iio
    import numpy as np
    import tempfile
    import os
    
    print("=== Large Image Sequence Generation ===")
    # Simulate processing large time-series data
    n_frames = 50
    height, width = 256, 256
    
    print(f"Generating {n_frames} frames of {height}x{width} images")
    
    with tempfile.TemporaryDirectory() as temp_dir:
        sequence_path = os.path.join(temp_dir, 'sequence.tiff')
        
        print("\n=== Batch Frame Generation ===")
        # Generate frames in memory (for demonstration)
        all_frames = []
        
        for frame_idx in range(n_frames):
            # Generate frame (simulate time-lapse data)
            frame = np.zeros((height, width), dtype=np.uint8)
            
            # Moving wave pattern
            x = np.arange(width)
            y = np.arange(height)
            X, Y = np.meshgrid(x, y)
            
            wave = np.sin(0.1 * X + frame_idx * 0.2) * np.cos(0.1 * Y + frame_idx * 0.15)
            frame = ((wave + 1) * 127.5).astype(np.uint8)
            
            # Add some noise
            noise = np.random.normal(0, 10, (height, width))
            frame = np.clip(frame.astype(float) + noise, 0, 255).astype(np.uint8)
            
            all_frames.append(frame)
            
            if frame_idx % 10 == 0:
                print(f"  Generated frame {frame_idx}: mean={frame.mean():.1f}, std={frame.std():.1f}")
        
        # Write all frames at once
        # is_batch=True writes each frame as its own TIFF series, so imiter yields one frame per series
        # and imread(index=i) selects frame i; without it the stack is written as a single (N, H, W) series.
        iio.imwrite(sequence_path, np.stack(all_frames), is_batch=True)
        print(f"Wrote {n_frames} frames to multi-page TIFF")
        
        print("\n=== Memory-Efficient Reading with imiter ===")
        # Use imiter for memory-efficient frame iteration (unique to imageio)
        
        # Process frames in batches without loading all into memory
        batch_size = 10
        batch_stats = []
        
        frames_processed = 0
        current_batch = []
        
        for frame in iio.imiter(sequence_path):
            current_batch.append(frame)
            frames_processed += 1
            
            if len(current_batch) == batch_size or frames_processed == n_frames:
                # Process batch
                batch_array = np.stack(current_batch)
                
                # Compute batch statistics
                stats = {
                    'batch_num': len(batch_stats) + 1,
                    'frames_in_batch': len(current_batch),
                    'mean_intensity': np.mean(batch_array),
                    'temporal_std': np.std(np.mean(batch_array, axis=(1, 2))),
                    'spatial_variation': np.mean(np.std(batch_array, axis=(1, 2)))
                }
                
                batch_stats.append(stats)
                print(f"Batch {stats['batch_num']}: {stats['frames_in_batch']} frames, "
                      f"mean={stats['mean_intensity']:.1f}, "
                      f"temporal_std={stats['temporal_std']:.2f}")
                
                current_batch = []
        
        print(f"\nProcessed {frames_processed} frames in {len(batch_stats)} batches using imiter")
        assert frames_processed == n_frames, f"imiter yielded {frames_processed} item(s), expected {n_frames} frames"
        
        print("\n=== Sequence Analysis ===")
        # Overall statistics
        total_mean = np.mean([s['mean_intensity'] for s in batch_stats])
        total_temporal_var = np.mean([s['temporal_std'] for s in batch_stats])
        
        print(f"Overall mean intensity: {total_mean:.2f}")
        print(f"Average temporal variation: {total_temporal_var:.3f}")
        
        # File size efficiency
        file_size_mb = os.path.getsize(sequence_path) / (1024 * 1024)
        per_frame_kb = (file_size_mb * 1024) / n_frames
        print(f"File size: {file_size_mb:.2f} MB ({per_frame_kb:.1f} KB/frame)")
        
        print("\n=== Selective Frame Access ===")
        # Demonstrate reading specific frames
        frame_10 = iio.imread(sequence_path, index=10)
        frame_25 = iio.imread(sequence_path, index=25)
        print(f"Frame 10 shape: {frame_10.shape}, mean: {frame_10.mean():.1f}")
        print(f"Frame 25 shape: {frame_25.shape}, mean: {frame_25.mean():.1f}")
        # TIFF is lossless: the frames read back must be the frames written
        assert np.array_equal(frame_10, all_frames[10]) and np.array_equal(frame_25, all_frames[25])
    
    print("Example 4: Memory-efficient streaming test passed")
    
except Exception as e:
    print(f"Example 4: Error in streaming test: {e}")
    import traceback
    traceback.print_exc()
    raise

## Example 5: improps and immeta for Metadata Handling

Demonstrates improps for standardized properties and immeta for format-specific metadata.

In [ ]:
# Example 5: improps and immeta for metadata handling
try:
    import imageio.v3 as iio
    import imageio
    import numpy as np
    import tempfile
    import os
    
    print("=== Format System (from imageio module) ===")
    # Access format information from main imageio module
    formats_list = list(imageio.formats)
    
    print(f"Total formats available: {len(formats_list)}")
    print(f"Sample formats: {[f.name for f in formats_list[:10]]}")
    
    # Find specific format info
    png_format = None
    for fmt in formats_list:
        if fmt.name == 'PNG-PIL':
            png_format = fmt
            break
    
    if png_format:
        print(f"\nPNG format info:")
        print(f"  Name: {png_format.name}")
        print(f"  Extensions: {png_format.extensions}")
        print(f"  Description: {png_format.description[:60]}...")
    
    print("\n=== Format-Specific Options ===")
    # Test different format-specific options
    test_image = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
    
    with tempfile.TemporaryDirectory() as temp_dir:
        # PNG with different compression levels
        png_files = {}
        for compress_level in [1, 6, 9]:
            png_path = os.path.join(temp_dir, f'test_compress_{compress_level}.png')
            iio.imwrite(png_path, test_image, compress_level=compress_level)
            file_size = os.path.getsize(png_path)
            png_files[compress_level] = file_size
            print(f"PNG compression {compress_level}: {file_size} bytes")
        
        # JPEG with different quality levels
        jpeg_files = {}
        for quality in [30, 70, 95]:
            jpg_path = os.path.join(temp_dir, f'test_quality_{quality}.jpg')
            iio.imwrite(jpg_path, test_image, quality=quality)
            file_size = os.path.getsize(jpg_path)
            jpeg_files[quality] = file_size
            print(f"JPEG quality {quality}: {file_size} bytes")
        
        print(f"\nPNG compression ratio (9 vs 1): {png_files[9] / png_files[1]:.2f}x")
        print(f"JPEG quality ratio (95 vs 30): {jpeg_files[95] / jpeg_files[30]:.2f}x")
    
    print("\n=== Metadata Handling with improps and immeta ===")
    # Create multi-channel TIFF with metadata
    z_stack_frames = []
    n_slices = 8
    
    for z in range(n_slices):
        # Simulate z-stack with varying focus
        frame = np.zeros((64, 64), dtype=np.uint8)
        
        # Create circular pattern that changes with z
        y, x = np.ogrid[:64, :64]
        center = 32
        radius = 15 + z * 2
        mask = (x - center)**2 + (y - center)**2 <= radius**2
        
        frame[mask] = 200 - z * 15  # Intensity changes with depth
        z_stack_frames.append(frame)
    
    print(f"Created {len(z_stack_frames)} z-stack slices")
    
    with tempfile.TemporaryDirectory() as temp_dir:
        # Save as multi-page TIFF
        zstack_path = os.path.join(temp_dir, 'zstack.tiff')
        
        # Add metadata
        metadata = {
            'description': 'Multi-page z-stack microscopy',
            'software': 'ImageIO test'
        }
        
        iio.imwrite(zstack_path, z_stack_frames, metadata=metadata)
        
        # Read back and check properties using improps (unique to imageio v3)
        props = iio.improps(zstack_path)
        print(f"\nimprops output:")
        print(f"  Shape: {props.shape}")
        print(f"  Dtype: {props.dtype}")
        print(f"  Is batch: {props.is_batch}")
        print(f"  N images: {props.n_images}")
        
        # Get format-specific metadata using immeta (unique to imageio v3)
        meta = iio.immeta(zstack_path)
        print(f"\nimmeta output (format-specific):")
        print(f"  Metadata keys: {list(meta.keys())[:8]}")
        if 'description' in meta:
            print(f"  Description: {meta['description']}")
        
        # Read individual pages and analyze
        print(f"\n=== Z-Stack Analysis ===")
        page_means = []
        
        for i, page in enumerate(iio.imiter(zstack_path)):
            page_mean = np.mean(page)
            page_means.append(page_mean)
            if i < 3:
                print(f"Slice {i}: shape={page.shape}, mean intensity={page_mean:.1f}")
        
        # Analyze z-stack progression
        intensity_gradient = np.diff(page_means)
        print(f"Z-stack mean intensity trend: {np.mean(intensity_gradient):.1f} per slice")
        print(f"Total intensity range: [{min(page_means):.1f}, {max(page_means):.1f}]")
    
    print("\n=== Extension-based Format Detection ===")
    # Demonstrate format selection
    with tempfile.TemporaryDirectory() as temp_dir:
        test_data = np.random.randint(0, 255, (32, 32, 3), dtype=np.uint8)
        
        # imageio automatically selects format based on extension
        test_formats = [
            ('image.png', 'PNG'),
            ('image.jpg', 'JPEG'),
            ('image.bmp', 'BMP'),
            ('image.tiff', 'TIFF')
        ]
        
        for filename, expected_fmt in test_formats:
            file_path = os.path.join(temp_dir, filename)
            iio.imwrite(file_path, test_data)
            
            # Verify file was created
            exists = os.path.exists(file_path)
            size = os.path.getsize(file_path) if exists else 0
            print(f"{filename}: created={exists}, size={size} bytes")
    
    print("Example 5: improps and immeta metadata handling test passed")
    
except Exception as e:
    print(f"Example 5: Error in metadata test: {e}")
    import traceback
    traceback.print_exc()
    raise